# cc-site-extract.ipynb

extracts course data from https://commoncore.hku.hk.  
outputs as JSON.  

## fields
- course name
- sustainable development goals
- semester offered
- study load distribution (and total study load)
- assignment distribution
- communication-intensive?

In [10]:
# extract course list from https://commoncore.hku.hk

import requests
from bs4 import BeautifulSoup

courses = set()

def scan_inquiry(inquiry):
	url = f"https://commoncore.hku.hk/{inquiry}/"
	print(f"fetching {url}...")

	response = requests.get(url)
	if response.status_code != 200:
		raise RuntimeError("bad response")
	
	soup = BeautifulSoup(response.text, "html.parser")

	for elem in soup.select(".elementor-widget-container:has(> h5)"):
		h5 = elem.find("h5")
		course_code = h5.get_text().strip()
		course_name = h5.find_next_sibling().get_text()
		
		print(f"{course_code}: {course_name}")

inquiries = [
	"scientific-and-technological-literacy",
	"arts-and-humanities",
	"global-issues",
	"china-culture-state-and-society",
	"artificial-intelligence",
]
for inquiry in inquiries:
	scan_inquiry(inquiry)

print(f"len={len(courses)}")


fetching https://commoncore.hku.hk/scientific-and-technological-literacy/...
CCST5035:  Making Sense of Science-related Social Issues (Online) 
CCST5037: Mathematics: A Cultural Heritage (Online)
CCST9003: Everyday Computing and the Internet
CCST9006: Chasing Biomedical Miracles: Promises and Perils
CCST9009: Living with Stem Cells
CCST9012: Our Place in the Universe
CCST9013: Our Living Environment
CCST9014: Science and Music
CCST9016: Energy: Its Evolution and Environmental Impacts
CCST9017: Hidden Order in Daily Life: A Mathematical Perspective
CCST9018: Origin and Evolution of Life
CCST9020: Sustainable Development of the Built Environment
CCST9022: How the Mass Media Depicts Science, Technology and the Natural World
CCST9023: The Oceans: Science and Society
CCST9025: Genetics and Human Nature
CCST9026: Scientific Revolutions: Their Continuing Impact on Our World and Society
CCST9027: The Science of Irrational Thinking
CCST9030: Forensic Science: Unmasking Evidence, Mysteries and C

In [11]:
# if sis-info.json already exists, search those courses as well

import os.path
import json

sis_path = "sis-info.json"

if os.path.isfile(sis_path):
	print(f"extracting from {sis_path}...")
	with open(sis_path, "r") as file:
		data = json.load(file)
		courses.update(map(lambda x : x.strip(), data.keys()))

print(f"len={len(courses)}")

extracting from sis-info.json...
len=318


In [16]:
# traverse course list and get course info per course

import concurrent.futures
import threading
import time
import traceback

courses = sorted(courses)

course_info = {}
course_info_lock = threading.Lock()
print_lock = threading.Lock()

fail_count = 0

def scan_course(course):
	url = f"https://commoncore.hku.hk/{course}/"

	for _ in range(3):
		with print_lock:
			print(f"fetching {url}... ")
		response = requests.get(url)
		if response.status_code == 200 or response.status_code == 404:
			break
		with print_lock:
			print("fail, retry... ")
		time.sleep(1)
	if response.status_code == 200:
		pass
	elif response.status_code == 404:
		with print_lock:
			print(f"{course} not found :(")
		return
	else:
		raise RuntimeError("bad response (%d)" % response.status_code)

	try:
		soup = BeautifulSoup(response.text, "html.parser")

		# course name
		temp = soup.select_one("td > h1 + h1")
		if temp:
			course_name = temp.get_text().strip().replace("\n", "")
		else:
			course_name = soup.select_one("td > h1 > * + *").next_sibling.string.strip()

		# sustainable development goals
		sdgs = [ elem["alt"] for elem in soup.select(".span6.zmt.sdgs-item > img") ]
		# print(sdgs)

		# semester offered
		temp = soup.find(id="osdt")
		if temp == None:
			temp = soup.find(id=f"{course}osdt")
		sem_offered = temp.find_next_sibling().get_text().strip()
		
		# print(sem_offered)

		# study load
		rows = soup.select_one("#load + table").select("tr")
		study_load_dist = {}
		for elem in rows[1:-1]:
			tds = elem.find_all("td")
			if len(tds) != 2:
				continue
			activity, hours = tds
			study_load_dist[activity.get_text()] = hours.get_text().strip()
		study_load_total = rows[-1].find_all("td")[-1].get_text().strip()
		# print(study_load_dist, study_load_total)

		# assessment
		rows = soup.select_one("#ass + table").select("tr")
		ass_dist = {}
		for elem in rows[1:]:
			tds = elem.find_all("td")
			if len(tds) != 2:
				continue
			task, weight = tds
			ass_dist[task.get_text().strip()] = weight.get_text().strip()
		# print(ass_dist)

		# communication-intensive?
		first_para_after_desc = soup.find(
			lambda elem: elem.get("class") == [".sdgs-text"]
				and "Communication-intensive Courses" in elem.text.strip()
		)
		is_comm_intensive = first_para_after_desc != None

		with course_info_lock:
			course_info[course] = {
				"course_name": course_name,
				"sdgs": sdgs,
				"sem_offered": sem_offered,
				"study_load_dist": study_load_dist,
				"study_load_total": study_load_total,
				"ass_dist": ass_dist,
				"is_comm_intensive": is_comm_intensive,
			}
			
		with print_lock:
			print(f"\033[32m{course}: [ok]\033[0m")
	except Exception as err:
		fail_count += 1
		with print_lock:
			print(f"\033[31m{course}: [error]")
			print(f"{traceback.format_exc()}\033[0m")

with concurrent.futures.ThreadPoolExecutor() as executor:
	executor.map(scan_course, courses)

print("done!")
print(f"{fail_count} failed")


fetching https://commoncore.hku.hk/CCCH5006/... 
fetching https://commoncore.hku.hk/CCCH5017/... 
fetching https://commoncore.hku.hk/CCCH5039/... 
fetching https://commoncore.hku.hk/CCCH5044/... 
fetching https://commoncore.hku.hk/CCCH5054/... 
fetching https://commoncore.hku.hk/CCCH5057/... 
CCCH5017: [ok]
fetching https://commoncore.hku.hk/CCCH6001/... 
CCCH5039: [ok]
fetching https://commoncore.hku.hk/CCCH9001/... 
CCCH5054: [ok]
fetching https://commoncore.hku.hk/CCCH9002/... 
CCCH5044: [ok]
fetching https://commoncore.hku.hk/CCCH9003/... 
CCCH5006: [ok]
fetching https://commoncore.hku.hk/CCCH9004/... 
CCCH9001: [ok]
fetching https://commoncore.hku.hk/CCCH9005/... 
CCCH9002: [ok]
fetching https://commoncore.hku.hk/CCCH9006/... 
CCCH9003: [ok]
fetching https://commoncore.hku.hk/CCCH9007/... 
CCCH9004: [ok]
fetching https://commoncore.hku.hk/CCCH9008/... 
CCCH5057: [ok]
fetching https://commoncore.hku.hk/CCCH9009/... 
CCCH6001: [ok]
fetching https://commoncore.hku.hk/CCCH9010/... 
CC

In [17]:
print(f"no. of courses: {len(course_info)}")

no. of courses: 270


In [18]:
import json

output_path = "cc-site-info.json"

with open(output_path, "w") as file:
	json.dump(course_info, file, indent=4)

print("done")

done
